In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q transformers torch python-telegram-bot

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
TOKEN = user_secrets.get_secret("Telegram bot token ")

from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

def analyze_sentiment(text):
    result = sentiment_model(text)[0]

    label = result["label"]
    score = result["score"]

    return label, round(score * 100, 2)


# Test the model
text = "This medicine is very effective and helped me a lot."

label, confidence = analyze_sentiment(text)

print("Sentiment:", label)
print("Confidence:", confidence, "%")

**model1try-1**

In [ ]:
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters
)

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text(
        "Welcome to Sentiment Analysis Bot! 😊\n\n"
        "Send me any text and I will analyze its sentiment."
    )


async def analyze_message(update: Update, context: ContextTypes.DEFAULT_TYPE):

    text = update.message.text

    label, confidence = analyze_sentiment(text)

    if label == "positive":
        emoji = "😊"

    elif label == "negative":
        emoji = "😞"

    else:
        emoji = "😐"

    response = (
        f"📝 Text: {text}\n\n"
        f"📊 Sentiment: {label.upper()} {emoji}\n"
        f"🎯 Confidence: {confidence}%"
    )

    await update.message.reply_text(response)


# Build application
app = ApplicationBuilder().token(TOKEN).build()

# Add handlers
app.add_handler(CommandHandler("start", start))

app.add_handler(
    MessageHandler(
        filters.TEXT & ~filters.COMMAND,
        analyze_message
    )
)

print("Telegram Bot Created Successfully!")

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

async def main():

    await app.initialize()

    await app.start()

    await app.updater.start_polling()

    print("🚀 Telegram Sentiment Bot is Running!")

    try:
        await asyncio.Event().wait()

    finally:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()


await main()

**With real dataset of drugs.com **

**model2try2**

In [ ]:
import pandas as pd
import numpy as np
import torch
import asyncio
import nest_asyncio

from transformers import pipeline
from telegram import Update
from telegram.ext import (
    ApplicationBuilder,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters
)

from kaggle_secrets import UserSecretsClient

# Load dataset
df = pd.read_csv("/kaggle/input/datasets/jessicali9530/kuc-hackathon-winter-2018/drugsComTest_raw.csv")

# Check required columns
required_columns = ["drugName", "review"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

df = df.dropna(subset=["drugName", "review"]).copy()

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0 if torch.cuda.is_available() else -1
)

def analyze_sentiment(text):

    result = sentiment_model(
        str(text)[:512],
        truncation=True
    )[0]

    label = result["label"].lower()

    return label, round(result["score"] * 100, 2)

In [ ]:
def normalize_sentiment(value):

    value = str(value).strip().lower()

    if value in ["positive", "pos", "1"]:
        return "positive"

    elif value in ["negative", "neg", "2"]:
        return "negative"

    elif value in ["neutral", "neu", "0"]:
        return "neutral"

    return None


if "sentiment" in df.columns:

    df["sentiment"] = df["sentiment"].apply(normalize_sentiment)

    # Predict missing or unrecognized sentiment values
    missing = df["sentiment"].isna()

    if missing.any():
        predictions = sentiment_model(
            df.loc[missing, "review"].astype(str).tolist(),
            batch_size=32,
            truncation=True,
            max_length=512
        )

        df.loc[missing, "sentiment"] = [
            p["label"].lower() for p in predictions
        ]

else:

    predictions = sentiment_model(
        df["review"].astype(str).tolist(),
        batch_size=32,
        truncation=True, 
        max_length=512
    )

    df["sentiment"] = [
        p["label"].lower() for p in predictions
    ]

print("Sentiment analysis completed!")
print(df["sentiment"].value_counts())

In [ ]:
def get_top_drugs(min_reviews=10, top_n=10):

    # Group reviews by drug
    results = df.groupby("drugName").agg(

        total_reviews=("review", "count"),

        positive_reviews=(
            "sentiment",
            lambda x: (x == "positive").sum()
        ),

        negative_reviews=(
            "sentiment",
            lambda x: (x == "negative").sum()
        ),

        neutral_reviews=(
            "sentiment",
            lambda x: (x == "neutral").sum()
        )

    ).reset_index()

    # Calculate positive percentage
    results["positive_percentage"] = (
        results["positive_reviews"] /
        results["total_reviews"] * 100
    )

    # Calculate negative percentage
    results["negative_percentage"] = (
        results["negative_reviews"] /
        results["total_reviews"] * 100
    )

    # Minimum number of reviews
    results = results[
        results["total_reviews"] >= min_reviews
    ]

    # Add average rating if available
    if "rating" in df.columns:

        ratings = df.groupby("drugName")["rating"].mean()

        results["average_rating"] = (
            results["drugName"].map(ratings)
        )

    # Rank drugs
    results = results.sort_values(
        by=["positive_percentage", "total_reviews"],
        ascending=[False, False]
    )

    return results.head(top_n)

In [ ]:
def get_drug_details(drug_name):

    drug_data = df[
        df["drugName"].str.lower() == drug_name.lower()
    ]

    if drug_data.empty:
        return None

    total = len(drug_data)

    positive = (
        drug_data["sentiment"] == "positive"
    ).sum()

    negative = (
        drug_data["sentiment"] == "negative"
    ).sum()

    neutral = (
        drug_data["sentiment"] == "neutral"
    ).sum()

    positive_percentage = positive / total * 100

    average_rating = None

    if "rating" in drug_data.columns:
        average_rating = pd.to_numeric(
            drug_data["rating"],
            errors="coerce"
        ).mean()

    return {
        "drug": drug_name,
        "total": total,
        "positive": positive,
        "negative": negative,
        "neutral": neutral,
        "positive_percentage": round(positive_percentage, 2),
        "average_rating": average_rating
    }

In [ ]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):

    message = """
💊 Welcome to Pharmaceutical Sentiment Analysis Bot!

Available Commands:

/analyze - Analyze patient review sentiment
/topdrugs - Show top-rated drugs
/drug DrugName - Analyze a specific drug

Send any patient review to analyze its sentiment.

Let's get started! 🚀
"""

    await update.message.reply_text(message)


# Analyze a patient review
async def analyze_command(update: Update, context: ContextTypes.DEFAULT_TYPE):

    await update.message.reply_text(
        "Please send your patient review as a message."
    )


# Show top drugs
async def topdrugs(update: Update, context: ContextTypes.DEFAULT_TYPE):

    results = get_top_drugs(min_reviews=10, top_n=10)

    if results.empty:
        await update.message.reply_text(
            "No drugs found with enough reviews."
        )
        return

    response = "🏆 TOP DRUGS BY PATIENT REVIEW SENTIMENT\n\n"

    for rank, (_, row) in enumerate(results.iterrows(), 1):

        response += (
            f"{rank}. 💊 {row['drugName']}\n"
            f"   Positive reviews: {row['positive_percentage']:.2f}%\n"
            f"   Total reviews: {row['total_reviews']}\n"
            f"   Negative reviews: {row['negative_percentage']:.2f}%\n"
        )

        if "average_rating" in results.columns:
            response += (
                f"   Average rating: {row['average_rating']:.2f}\n"
                if pd.notna(row["average_rating"])
                else "   Average rating: N/A\n"
            )

        response += "\n"

    response += (
        "Note: This ranking reflects patient-review sentiment only. "
        "It is not a clinical safety or efficacy recommendation."
    )

    await update.message.reply_text(response)


# Analyze a specific drug
async def drug_details(update: Update, context: ContextTypes.DEFAULT_TYPE):

    if not context.args:

        await update.message.reply_text(
            "Please enter a drug name.\nExample: /drug Prozac"
        )
        return

    drug_name = " ".join(context.args)

    result = get_drug_details(drug_name)

    if result is None:

        await update.message.reply_text(
            f"Sorry, {drug_name} was not found in the dataset."
        )
        return

    response = (
        f"💊 DRUG ANALYSIS: {result['drug']}\n\n"
        f"📊 Total Reviews: {result['total']}\n"
        f"😊 Positive: {result['positive']}\n"
        f"😞 Negative: {result['negative']}\n"
        f"😐 Neutral: {result['neutral']}\n\n"
        f"📈 Positive Percentage: "
        f"{result['positive_percentage']}%\n"
    )

    if result["average_rating"] is not None:
        response += (
            f"⭐ Average Rating: "
            f"{result['average_rating']:.2f}\n"
        )

    response += (
        "\nThis is a patient-review analysis, "
        "not a medical recommendation."
    )

    await update.message.reply_text(response)


# Automatically analyze every incoming text
async def analyze_message(update: Update, context: ContextTypes.DEFAULT_TYPE):

    text = update.message.text

    try:

        label, confidence = analyze_sentiment(text)

        emoji = {
            "positive": "😊",
            "negative": "😞",
            "neutral": "😐"
        }.get(label, "📊")

        response = (
            f"📝 Review:\n{text}\n\n"
            f"📊 Sentiment: {label.upper()} {emoji}\n"
            f"🎯 Confidence: {confidence}%"
        )

        await update.message.reply_text(response)

    except Exception as e:

        await update.message.reply_text(
            f"Error analyzing review: {str(e)}"
        )

In [ ]:
app = ApplicationBuilder().token(TOKEN).build()

# Register commands
app.add_handler(CommandHandler("start", start))

app.add_handler(CommandHandler("analyze", analyze_command))

app.add_handler(CommandHandler("topdrugs", topdrugs))

app.add_handler(CommandHandler("drug", drug_details))

# Register text messages
app.add_handler(
    MessageHandler(
        filters.TEXT & ~filters.COMMAND,
        analyze_message
    )
)

print("Bot initialized successfully!")

In [ ]:
nest_asyncio.apply()

async def main():

    await app.initialize()

    await app.start()

    await app.updater.start_polling()

    print("🚀 Pharmaceutical Sentiment Bot is Running!")

    try:
        await asyncio.Event().wait()

    finally:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()


await main()

In [ ]:
import os
import signal
import subprocess
import asyncio
import nest_asyncio

nest_asyncio.apply()

# Stop previous bot processes
current_pid = os.getpid()

result = subprocess.run(
    ["pgrep", "-f", "python"],
    capture_output=True,
    text=True
)

for pid in result.stdout.splitlines():
    pid = int(pid)

    if pid != current_pid:
        try:
            cmd = subprocess.check_output(
                ["ps", "-p", str(pid), "-o", "args="],
                text=True
            )

            # Change this to a unique part of your bot script name
            if "your_bot_script.py" in cmd:
                os.kill(pid, signal.SIGTERM)
                print(f"Stopped old bot process: {pid}")

        except (ProcessLookupError, subprocess.CalledProcessError):
            pass


# Start the new bot
async def main():

    await app.initialize()
    await app.start()
    await app.updater.start_polling()

    print("🚀 Pharmaceutical Sentiment Bot is Running!")

    try:
        await asyncio.Event().wait()

    finally:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()


await main()